# Opportunity · Lot-Sale Planner

You need to **raise cash** — a down payment, a tax bill, a rebalance. *Which specific lots
you sell* can swing the tax bill by thousands, because every lot has its own cost basis
and holding period. This picks the lots that raise your target at **minimum tax**, and
shows what naive selling (FIFO) would have cost instead.

Brokers default to FIFO unless you specify lots at the time of sale — this is the
spec-ID worklist.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from invest import config, analysis as A, ledger, mapping, opportunities as O

px.defaults.template = "plotly_white"
PALETTE = px.colors.qualitative.Safe

positions = pd.read_parquet(config.POSITIONS_PARQUET)
history = pd.read_parquet(config.PRICES_PARQUET)
transactions = pd.read_parquet(config.TRANSACTIONS_PARQUET)
entries, _, _ = ledger.load()
account_meta = mapping.load_account_meta()
lots = O.price_lots(ledger.lots_dataframe(entries), positions)
TODAY = pd.Timestamp.today().normalize()
print(f"{len(lots)} tax lots loaded · as of {TODAY:%Y-%m-%d}")

## Set the ask

`RAISE_AMOUNT` = cash to raise. `SYMBOL` = restrict to one holding, or `None` for the
whole portfolio. Rates are your marginal **long-term** and **short-term** rates. By
default we only sell from **taxable** accounts (selling in an IRA/401k raises no
current tax but isn't "free" — it's a withdrawal).

In [ ]:
RAISE_AMOUNT = 50_000
SYMBOL       = None          # e.g. "NVDA" to raise from one holding, or None for any
LT_RATE      = O.DEFAULT_LT_RATE   # long-term cap-gains + NIIT (edit to your bracket)
ST_RATE      = O.DEFAULT_ST_RATE   # short-term = ordinary income

plan = O.plan_lot_sale(lots, account_meta, raise_amount=RAISE_AMOUNT, symbol=SYMBOL,
                       lt_rate=LT_RATE, st_rate=ST_RATE, today=TODAY, strategy="min_tax")
tot = O.sale_plan_totals(plan, RAISE_AMOUNT)
if tot["shortfall"] > 0:
    print(f"⚠ Only ${tot['proceeds']:,.0f} available — ${tot['shortfall']:,.0f} short of the target"
          + (f" in {SYMBOL}." if SYMBOL else " in taxable accounts."))
print(f"Raise ${tot['proceeds']:,.0f}  ·  realize ${tot['gain']:,.0f} gain  ·  "
      f"tax ${tot['tax']:,.0f}  ·  effective rate {tot['effective_rate']:.1%}")

## The plan — which lots to sell

Lots chosen in tax-efficient order (losses first, then lowest-tax gains). Green bars
*reduce* your tax (realized losses); red bars add to it. The last lot is partially filled
to hit the target exactly.

In [ ]:
plan = plan.reset_index(drop=True)
plan["label"] = plan["symbol"] + " · " + plan["term"] + " · " + plan["acquired"].dt.strftime("%Y-%m")                 + " (" + plan["account_name"] + ")"
fig = px.bar(plan, x="proceeds", y="label", orientation="h", color="sell_tax",
             color_continuous_scale="RdYlGn_r", color_continuous_midpoint=0,
             hover_data={"sell_qty": ":.3f", "sell_gain": ":$,.0f", "sell_tax": ":$,.0f"},
             title=f"Sell these lots to raise ${RAISE_AMOUNT:,.0f}")
fig.update_layout(height=max(360, 32 * len(plan)), xaxis_tickprefix="$",
                  yaxis_title="", yaxis_autorange="reversed",
                  coloraxis_colorbar_title="tax $")
fig.show()

cols = ["account_name", "symbol", "term", "acquired", "sell_qty", "proceeds",
        "sell_gain", "sell_tax"]
disp = plan[cols].copy()
disp["acquired"] = disp["acquired"].dt.strftime("%Y-%m-%d")
display(disp.style.format({"sell_qty": "{:.3f}", "proceeds": "${:,.0f}",
        "sell_gain": "${:,.0f}", "sell_tax": "${:,.0f}"})
        .background_gradient(subset=["sell_tax"], cmap="RdYlGn_r"))

## What tax-aware selection saves you

The same \$ raised under four orderings: **min-tax** (this plan), **HIFO** (highest cost
first — a common broker option), **FIFO** (the usual default), and **max-gain** (worst
case). The gap between FIFO and min-tax is real money left on the table by not specifying
lots.

In [ ]:
cmp = O.compare_sale_strategies(lots, account_meta, raise_amount=RAISE_AMOUNT,
                                symbol=SYMBOL, lt_rate=LT_RATE, st_rate=ST_RATE, today=TODAY)
names = {"min_tax": "Min-tax (this plan)", "hifo": "HIFO", "fifo": "FIFO (default)", "max_gain": "Max-gain"}
cmp = cmp.rename(index=names)
fig = px.bar(cmp.reset_index(), x="index", y="tax", color="tax",
             color_continuous_scale="RdYlGn_r", color_continuous_midpoint=0, text="tax",
             title="Tax to raise the same cash, by lot-selection strategy")
fig.update_traces(texttemplate="$%{text:,.0f}", textposition="outside")
fig.update_layout(height=420, yaxis_tickprefix="$", xaxis_title="", yaxis_title="tax",
                  coloraxis_showscale=False)
fig.show()
saved = cmp.loc["FIFO (default)", "tax"] - cmp.loc["Min-tax (this plan)", "tax"]
print(f"Specifying lots saves ${saved:,.0f} vs the FIFO default on this ${RAISE_AMOUNT:,.0f} raise.")
display(cmp[["proceeds", "gain", "tax", "effective_rate", "tax_vs_min"]].style.format({
    "proceeds": "${:,.0f}", "gain": "${:,.0f}", "tax": "${:,.0f}",
    "effective_rate": "{:.1%}", "tax_vs_min": "${:+,.0f}"}))

## Raising from a single holding?

Set `SYMBOL` above to trim one position tax-efficiently (e.g. de-risking NVDA). Here's the
per-holding picture — available taxable value and the blended tax to liquidate it
entirely — so you can see where cash is cheapest to raise.

In [ ]:
tx_lots = O.attach_account_tax(lots, account_meta)
tx_lots = tx_lots[tx_lots["tax_treatment"] == "taxable"].copy()
tx_lots["days_held"] = (TODAY - tx_lots["acquired"]).dt.days
tx_lots["term"] = np.where(tx_lots["days_held"] > O.LONG_TERM_DAYS, "long", "short")
tx_lots["gain"] = tx_lots["market_value"] - tx_lots["cost_basis"]
tx_lots["tax"] = tx_lots["gain"] * np.where(tx_lots["term"] == "long", LT_RATE, ST_RATE)
by_sym = tx_lots.groupby("symbol").agg(
    taxable_value=("market_value", "sum"), unrealized=("gain", "sum"),
    tax_to_liquidate=("tax", "sum"))
by_sym["blended_rate"] = by_sym["tax_to_liquidate"] / by_sym["taxable_value"]
display(by_sym.sort_values("blended_rate").style.format({
    "taxable_value": "${:,.0f}", "unrealized": "${:,.0f}",
    "tax_to_liquidate": "${:,.0f}", "blended_rate": "{:.1%}"})
    .background_gradient(subset=["blended_rate"], cmap="RdYlGn_r"))

---
*Candidate generator, not advice. Tax rates are placeholders — set them to your bracket.
Lots, cost basis and acquisition dates come from the curated ledger; **confirm against
your broker's own basis** before placing any trade, and mind wash-sale rules when you
realize losses.*